In [ ]:
import os
import glob
import pandas as pd

# ======================================================
# EDIT THESE
# ======================================================
ROOT_DIR = r"C:\Users\Vivian\Documents\CONCH\conch_img_feats"
CSV_NAME = "fold_metrics_patient_20x40x_topk64.csv"

OUT_GLOBAL = os.path.join(ROOT_DIR, "patient_agg_summary_ALL_EXPERIMENTS.csv")

KEEP_MODELS = {
    "patient_max_20x40x_topk64",
    "patient_mean_20x40x_topk64",
    "patient_noisy_or_20x40x_topk64",
}

METRICS = ["auc", "balacc", "acc"]

# ======================================================
def load_and_summarize(csv_path: str):
    df = pd.read_csv(csv_path)

    need = {"fold", "level", "model", *METRICS}
    if not need.issubset(df.columns):
        return None

    df["level"] = df["level"].astype(str).str.lower().str.strip()
    df["model"] = df["model"].astype(str).str.strip()

    df = df[(df["level"] == "patient") & (df["model"].isin(KEEP_MODELS))].copy()
    if df.empty:
        return None

    for m in METRICS:
        df[m] = pd.to_numeric(df[m], errors="coerce")

    df = df.dropna(subset=METRICS)
    df = df.sort_values(["model", "fold"])
    df = df.drop_duplicates(subset=["model", "fold"], keep="last")

    summary = (
        df.groupby("model")[METRICS]
          .agg(["mean", "std"])
          .round(6)
    )

    return summary, df


# ======================================================
# Main
# ======================================================
all_rows = []
found = False

for csv_path in glob.glob(os.path.join(ROOT_DIR, "**", CSV_NAME), recursive=True):
    found = True
    exp_dir = os.path.basename(os.path.dirname(csv_path))

    out = load_and_summarize(csv_path)
    if out is None:
        continue

    summary, df_used = out

    # Save per-experiment summary
    out_csv = os.path.join(os.path.dirname(csv_path), "patient_agg_summary_mean_std.csv")
    summary.to_csv(out_csv)

    print(f"\n=== {exp_dir} ===")
    print(summary)

    # Build global table
    tidy = summary.copy()
    tidy.columns = [f"{m}_{s}" for m, s in tidy.columns]
    tidy = tidy.reset_index()
    tidy.insert(0, "experiment", exp_dir)
    all_rows.append(tidy)

if not found:
    raise FileNotFoundError(f"No '{CSV_NAME}' found under {ROOT_DIR}")

# ======================================================
# Save global summary
# ======================================================
if all_rows:
    global_df = pd.concat(all_rows, ignore_index=True)
    global_df.to_csv(OUT_GLOBAL, index=False)

    print("\n=== Global patient aggregation summary (all experiments) ===")
    print(global_df)
    print(f"\nSaved global summary to:\n{OUT_GLOBAL}")
else:
    print("No valid patient-level rows found.")
